In [ ]:
using LinearAlgebra
using May7Project
using DifferentialEquations
using Plots
using OrdinaryDiffEqLowOrderRK # for Euler's method
using OrdinaryDiffEqSDIRK # for implicit Euler

# Heat Equation with Explicit Euler
Try using explicit Euler to time step the finite difference approximation of the heat equation.

In [ ]:
function heat!(du, u, p, t)
    alpha = p[1];
    dx = p[2];
    x = p[3];
    ga = p[4];
    gb = p[5];
    f = p[6];
    Dxx = p[7];
    
    du .= alpha * Dxx * u + f.(x,t);
    # apply boundary conditions
    du[1] += alpha/dx^2 * ga(t);
    du[end] += alpha/dx^2 * gb(t);
    
    du
end


In [ ]:
x = LinRange(-5.0, 5.0, 501)[2:end-1];

u0 = exp.(-x.^2) # initial condition



tspan = (0.0, 5.0) # (t0, tmax)

alpha = 1;  # diffusivity
dx = x[2] - x[1];
n = length(x);
ga = t-> 0;
gb = t-> 0;
f = (x,t) -> 0; # source/sink
Dxx = sparse_second_derivative_matrix(n, dx);

p = (alpha, dx, x, ga, gb, f, Dxx);

It turns out that for a stable solution with explicit Euler, we need
$$
\frac{2\alpha \delta t}{\delta x^2}\leq 1.
$$
You can confirm this by experimenting with the various parameters here.

In [ ]:
# define and solve problem
prob = ODEProblem(heat!, u0, tspan, p)
dt = 0.0001;
@show 2 * alpha * dt/dx^2;

sol = solve(prob, Euler(), dt=dt, adaptive=false);

In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 21);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol(t_plot[i]), label="t=$(round(t_plot[i], digits=2))")
    ylims!(0, 1)
end
gif(anim, fps=3)

For explicit Euler, we need a small time step.

# Heat Equation with Implicit Euler
In congrast, implicit Euler can take arbitrarily large time steps, and not blow up (though they will have a bigger error).


In [ ]:
# define and solve problem
prob = ODEProblem(heat!, u0, tspan, p)
dt = 0.1;
sol = solve(prob, ImplicitEuler(), dt=dt, adaptive=false);

t_plot = LinRange(tspan[1], tspan[2], 21);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol(t_plot[i]), label="t=$(round(t_plot[i], digits=2))")
    ylims!(0, 1)
end
gif(anim, fps=3)

Implicit Euler allows for large time steps.

# BVPs
Consider solving the two point boundary value problem
$$
-u''= f(x), \quad a<x<b,
$$
subject to the boundary conditions:
$$
u(a) = g_a,\quad u(b) = g_b
$$

## Parabolic Example
The function $x(1-x)$ satisfies $-u'' = 2$ on $(0,1)$, with $u(0) = u(1) = 0$.  This finite difference scheme recovers it exacly (up to floating point).

In [ ]:
a = 0.0;
b = 1.0;
nx= 100 + 1; # number points, including endpoints
N = nx - 2; # number of interior points

x = LinRange(a, b, nx)[2:end-1]; # only interior points
@show Δx = x[2] - x[1];

Dxx = sparse_second_derivative_matrix(N, Δx);
A = -Dxx;

f = x-> 2;
rhs = f.(x); # only evaluate at intetior points;

u = A\rhs;

@show norm(u - x .*(1 .-x), 2);

plot(x, u,marker=:circle, label="FD Solution")
plot!(x, x .*(1 .-x), label="Exact Solution")
xlabel!("x")
title!("Finite Difference Solution with N = $N")


This is exact up to floating point

## General Example
Let us manufacture a solution by letting $u^\dagger(x) = e^x$, and then finding the $g_a$, $g_b$, and $f(x)$, such that $-\partial_{xx} u^\dagger = f(x)$, and satisfies the boundary conditions.

For $u^\dagger = e^x$, $f = -e^x$, and $g_a = e^a$, while $g_b = e^b$.  This has finite error at all discretizations.

In [ ]:
a = -1.0;
b = 2.0;
N = 40 + 1; # number points, including endpoints
nx = N - 2; # number of interior points

x = LinRange(a, b, N)[2:end-1]; # only interior points
@show Δx = x[2] - x[1];

Dxx = sparse_second_derivative_matrix(nx, Δx);
A = -Dxx;



f = x-> -exp(x);
ga = exp(a);
gb = exp(b);
rhs = f.(x); # only evaluate at intetior points;
rhs[1] += ga/Δx^2;
rhs[end] += gb/Δx^2;

u = A\rhs;
@show norm(u - exp.(x), 2);

plot(x, u, marker=:circle, label="FD Solution")
plot!(x, exp.(x), label="Exact Solution")
xlabel!("x")
title!("Finite Difference Solution with N = $N")

Try convergence testing in the $\infty$ - norm:

In [ ]:
a =-1.0;
b = 2.0;

nx_vals = [5, 10, 20, 40, 80, 160, 320, 640] .+ 1; # number points, including endpoints
h_vals = (b - a) ./ (nx_vals .- 1);
errors = zeros(length(nx_vals));
for (i,nx) in enumerate(nx_vals)
    N = nx - 2; # number of points, including endpoints
    x = LinRange(a, b, nx)[2:end-1]; # only interior points
    Δx = x[2] - x[1];
    Dxx = sparse_second_derivative_matrix(N, Δx);
    A = -Dxx;
    f = x-> -exp(x);
    ga = exp(a);
    gb = exp(b);
    rhs = f.(x); # only evaluate at intetior points;
    rhs[1] += ga/Δx^2;
    rhs[end] += gb/Δx^2;
    # solve the problem
    u = A\rhs;
    # compute error in ∞-norm
    errors[i] = norm(u - exp.(x), Inf);
end

scatter(h_vals, errors, xscale=:log10, yscale=:log10,label="Error")
plot!(h_vals, 0.5 * h_vals.^2, label="O(δx²)", linestyle=:dash)
plot!(h_vals, 0.1 * h_vals, label="O(δx)", linestyle=:dash)
xlabel!("δx")
ylabel!("Error")

To compare against 2-norm, we need to scale by $\sqrt{\delta x}$, so that there is a well defined limit as $n_x\to \infty$.  This amounts to studying
$$
\sqrt{\int_a^b |(\mathcal{I}\mathbf{u})(x) -u^\dagger(x)|^2 dx}\approx \sqrt{\delta x \cdot \sum_{j=1}^{n_x-1}|u_j - u(x_j)|^2}
$$
This also converges at an $\mathrm{O}(\delta x^2)$ rate.  

In [ ]:
a =-1.0;
b = 2.0;

nx_vals = [5, 10, 20, 40, 80, 160, 320, 640] .+ 1; # number points, including endpoints
h_vals = (b - a) ./ (nx_vals .- 1);
errors = zeros(length(nx_vals));
for (i,nx) in enumerate(nx_vals)
    N = nx - 2; # number of points, including endpoints
    x = LinRange(a, b, nx)[2:end-1]; # only interior points
    Δx = x[2] - x[1];
    Dxx = sparse_second_derivative_matrix(N, Δx);
    A = -Dxx;
    f = x-> -exp(x);
    ga = exp(a);
    gb = exp(b);
    rhs = f.(x); # only evaluate at intetior points;
    rhs[1] += ga/Δx^2;
    rhs[end] += gb/Δx^2;
    # solve the problem
    u = A\rhs;
    # compute error in 2-norm
    errors[i] = sqrt(Δx) * norm(u - exp.(x), 2);
end

scatter(h_vals, errors, xscale=:log10, yscale=:log10,label="Error")
plot!(h_vals, 0.5 * h_vals.^2, label="O(δx²)", linestyle=:dash)
plot!(h_vals, 0.1 * h_vals, label="O(δx)", linestyle=:dash)
xlabel!("δx")
ylabel!("Error")